# Churn Modeling Pipeline and Evaluation

This notebook compares baseline and non-linear models using a reproducible preprocessing + modeling pipeline.

**Objective:** Build a practical churn classifier for retention prioritization while avoiding leakage.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve().parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from churn_pipeline.pipeline import load_data, run_model_comparison, save_model_diagnostics

DATA_PATH = ROOT / "data" / "raw" / "telco_churn.csv"
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = load_data(DATA_PATH)
scores, artifacts, bundle = run_model_comparison(df)
scores

In [ ]:
best_model_name = artifacts["best_model_name"]
best_model = artifacts["best_model"]
best_preds = artifacts["predictions"][best_model_name]

save_model_diagnostics(
    model=best_model,
    model_name=best_model_name,
    y_true=bundle.y_test,
    y_pred=best_preds["y_pred"],
    y_prob=best_preds["y_prob"],
    output_dir=FIG_DIR,
)

print("Best model:", best_model_name)

In [ ]:
for name in [
    "confusion_matrix_best_model.png",
    "roc_curve_best_model.png",
    "top_feature_importance.png",
]:
    path = FIG_DIR / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

## Modeling Notes

- Comparison focuses on Logistic Regression (interpretable baseline) and Random Forest (non-linear benchmark).
- Evaluation emphasizes recall/F1/AUC, since missing likely churners has direct business cost.
- Leakage columns (for example churn reason/category) are intentionally excluded from training.